# Gemma 3 1B tool-call QLoRA
Run in a Colab GPU runtime. Set `HF_TOKEN` in Colab secrets after accepting Gemma and dataset terms. Save outputs to Drive. This notebook is an experiment workflow; it does not contain measured findings.


In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
import os
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
%cd /content
!git clone https://github.com/REPLACE_WITH_YOUR_REPOSITORY/gemma-tool-calling-lab.git project
%cd /content/project
!pip install -r requirements-train.txt


Replace the clone URL with this repository's URL, or upload the repository folder into Colab. Keep the token in Colab secrets, not notebook source.


In [ ]:
!python scripts/prepare_data.py --output-dir /content/drive/MyDrive/gemma-tool-lab/data/processed
import json, pathlib
data_dir=pathlib.Path('/content/drive/MyDrive/gemma-tool-lab/data/processed')
print(json.loads((data_dir/'split_manifest.json').read_text())['actual_sizes'])


In [ ]:
import torch
from transformers import AutoTokenizer
from scripts.common import chat_prompt
cfg=json.load(open('configs/train.json'))
tokenizer=AutoTokenizer.from_pretrained(cfg['base_model'], token=os.environ['HF_TOKEN'])
sample=json.loads((data_dir/'train.jsonl').read_text().splitlines()[0])
prompt=chat_prompt(tokenizer,sample['question'],sample['tools'])
answer=json.dumps(sample['target'],ensure_ascii=False)+tokenizer.eos_token
print(tokenizer.decode(tokenizer(prompt+answer)['input_ids']))
print('Prompt tokens are masked; answer token count:',len(tokenizer(answer, add_special_tokens=False)['input_ids']))


## Smoke test then full training
The next cell configures assistant-only labels, QLoRA and a 20-step smoke run. Review decoded sample and loss mask before starting. Validate the saved smoke adapter in a clean vLLM serving environment before setting `RUN_FULL=True`.


In [ ]:
# Training code intentionally uses the same prompt_text/chat template contract as preparation and inference.
# Run scripts/train_qlora.py --max-steps 20 --output-dir ... for the smoke test.
# After successful adapter load and nonempty vLLM generation, set RUN_FULL=True.
RUN_FULL=False
if RUN_FULL:
    !python scripts/train_qlora.py --output-dir /content/drive/MyDrive/gemma-tool-lab/adapter
else:
    print('Smoke step: implement/run the command below; full training is gated until serving validation.')
    !python scripts/train_qlora.py --max-steps 20 --output-dir /content/drive/MyDrive/gemma-tool-lab/smoke_adapter
